# Quantum Battleship (Zeno-based scanner)

- Cleaned & integrated version
- Teaching/demo cells removed
- Includes: Board, Rules/Score, Quantum Engine, Game, UI, and a driver cell

Tips:
- If you don't have `qiskit-aer` installed, the engine will fall back to a simple probabilistic model.
- Start with small `shots`/`zeno_steps` (e.g., 200/20) to validate flow.

# Install Packages


In [ ]:
# Add packages which are needed to be installed here.
!pip install qiskit[visualization] qiskit-ibm-runtime qiskit-aer qiskit_qasm3_import numpy ipywidgets
!jupyter nbextension enable --py widgetsnbextension


Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK


# Import, Rules & Scores


In [ ]:

# === Imports & Data Models ===
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import math
import random
import numpy as np

# Optional Qiskit imports (fallback to a simulator if unavailable)
_qiskit_ok = True
try:
    from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
    try:
        from qiskit_aer import AerSimulator
        _aer_ok = True
    except Exception:
        _aer_ok = False
    # If Aer not available, using qiskit's BasicAer isn't ideal; we will simulate ourselves instead.
except Exception:
    _qiskit_ok = False
    _aer_ok = False


@dataclass
class Outcome:
    label: str                     # 'found' | 'miss' | 'explodes'
    p_found: float = 0.0
    p_explodes: float = 0.0
    counts: Optional[Dict[str, int]] = None


@dataclass
class Cost:
    meas: int = 0
    twoq: int = 0
    shots: int = 0
    fires: int = 0


@dataclass
class Spent:
    meas: int = 0
    twoq: int = 0


@dataclass
class Rules:
    # Scoring weights / budgets
    reward_hit: int = 15
    penalty_miss: int = -15

    penalty_explosion: int = -0.1
    meas_cost_weight: float = -0.0005   # per measurement
    twoq_cost_weight: float = -0.01    # per 2-qubit gate (if you add them later)

    meas_limit: int = 1_000_000
    twoq_limit: int = 1_000_000

    def overBudget(self, spent: Spent) -> bool:
        return spent.meas > self.meas_limit or spent.twoq > self.twoq_limit


@dataclass
class Score:
    ai_points: float = 0.0
    user_points: float = 0.0

    ai_usage = {
        'shots': 0,
        'measuments': 0,
        'positives': 0,
        'explosions': 0,
        'fire times': 0,
        'hits': 0,
        'misses': 0,
    }

    user_usage = {
        'shots': 0,
        'measuments': 0,
        'positives': 0,
        'explosions': 0,
        'fire times': 0,
        'hits': 0,
        'misses': 0,
    }

    spent: Spent = field(default_factory=Spent)

    def apply(self, ai, outcome: Outcome, cost: Cost, rules: Rules):
        # Tally costs
        self.spent.meas += cost.meas
        self.spent.twoq += cost.twoq

        #
        if ai:
            points = self.ai_points
            usage = self.ai_usage
        else:
            points = self.user_points
            usage = self.user_usage

        # Outcome-based scoring
        if outcome.label == 'found':
            points += rules.reward_hit
        elif outcome.label == 'miss':
            points += rules.penalty_miss
        elif outcome.label == 'explodes':
            points += rules.penalty_explosion * outcome.p_explodes

        # Add records
        usage['shots'] += cost.shots
        usage['fire times'] += cost.fires
        if cost.fires == 1:
            usage['hits'] += outcome.p_found
            usage['misses'] += outcome.p_explodes
        usage['measuments'] += cost.meas
        usage['positives'] += cost.shots * outcome.p_found
        usage['explosions'] += cost.shots * outcome.p_explodes

        # Operation costs
        points += rules.meas_cost_weight * cost.meas
        points += rules.twoq_cost_weight * cost.twoq

        if ai:
            self.ai_points = points
        else:
            self.user_points = points


# Class Board

In [ ]:

# === Board ===
class Board:
    def __init__(self, size: int = 10):
        self.size = size
        # 'O' empty, 'M' miss, 'X' hit
        self.opponent_board = [['O' for _ in range(size)] for _ in range(size)]
        self.player_board = [['O' for _ in range(size)] for _ in range(size)]
        # Ships as sets of (r, c)
        self.opponent_ships: List[set] = []
        self.player_ships: List[set] = []

    def place_ship(self, side: str = "opponent", seed: Optional[int] = None, count: int = 5, length: int = 1):
        """Place 'count' ships of given 'length' (default single-cell ships) randomly."""
        if seed is not None:
            rnd = random.Random(seed)
        else:
            rnd = random

        ships_list = self.opponent_ships if side == "opponent" else self.player_ships
        board = self.opponent_board if side == "opponent" else self.player_board

        placed = 0
        attempts = 0
        while placed < count and attempts < 10_000:
            attempts += 1
            # random orientation for length>1
            vertical = rnd.choice([True, False])
            if vertical:
                r = rnd.randint(0, self.size - length)
                c = rnd.randint(0, self.size - 1)
                coords = {(r+i, c) for i in range(length)}
            else:
                r = rnd.randint(0, self.size - 1)
                c = rnd.randint(0, self.size - length)
                coords = {(r, c+i) for i in range(length)}

            # Check overlap
            if any(coord in cell for ship in ships_list for cell in [ship] for coord in coords):
                continue
            ships_list.append(coords)
            placed += 1

    def all_ships_cells(self, side: str = "opponent") -> set:
        ships_list = self.opponent_ships if side == "opponent" else self.player_ships
        cells = set()
        for s in ships_list:
            cells |= s
        return cells

    # [ADD] peek only, do not modify board (for scanning logic)
    def peek_has_ship(self, r: int, c: int, side: str = "opponent") -> bool:
        ships = self.opponent_ships if side == "opponent" else self.player_ships
        return any((r, c) in s for s in ships)

    # 'has_ship' here *fires* at (r, c): modifies board to X/M and returns True if hit
    def has_ship(self, r: int, c: int, side: str = "opponent") -> bool:
        ships = self.opponent_ships if side == "opponent" else self.player_ships
        board = self.opponent_board if side == "opponent" else self.player_board
        hit = False
        for s in ships:
            if (r, c) in s:
                hit = True
                break
        # reveal on fire
        if hit:
            board[r][c] = 'X'
        else:
            board[r][c] = 'M'
        return hit

    # [ADD] reveal after probing (non-destructive scan result mapping)
    def reveal(self, r: int, c: int, label: str, side: str = "opponent") -> None:
        board = self.opponent_board if side == "opponent" else self.player_board
        if label == 'found' or label == 'explodes':
            board[r][c] = 'S'
        elif label == 'no_ship':
            board[r][c] = 'NS'


# Quantum Engine

In [ ]:

# === QuantumEngine (Zeno) ===
class QuantumEngine1:
    def __init__(self, real_backend: bool = False):
        # self.use_qiskit = (_qiskit_ok and _aer_ok and real_backend)
        self.use_qiskit = True
        if self.use_qiskit:
            self.backend = AerSimulator(method='automatic')
        else:
            self.backend = None  # we'll use a probabilistic approximation

    # [ADD] Wilson score interval for a proportion
    def _wilson_ci(self, k: int, n: int, z: float = 1.96):
        if n <= 0:
            return (0.0, 0.0)
        p = k / n
        denom = 1.0 + z*z/n
        centre = (p + z*z/(2*n)) / denom
        half = z * math.sqrt( (p*(1-p)/n) + (z*z/(4*n*n)) ) / denom
        return (max(0.0, centre - half), min(1.0, centre + half))

    def _zeno_counts_qiskit(self, shots: int, zeno_steps: int, has_ship: bool):
        # Build circuit: 1 qubit, zeno_steps mid-measurements when has_ship, final readout
        qr = QuantumRegister(1, 'q')
        cr = ClassicalRegister(zeno_steps + 1, 'm')
        qc = QuantumCircuit(qr, cr)
        theta = math.pi / (2 * max(1, zeno_steps))

        for i in range(zeno_steps):
            qc.ry(2 * theta, qr[0])
            if has_ship:
                qc.measure(qr[0], cr[i])
                qc.reset(qr[0])
            qc.barrier()
        qc.measure(qr[0], cr[zeno_steps])

        tqc = transpile(qc, self.backend)
        res = self.backend.run(tqc, shots=shots).result()
        counts = res.get_counts()

        exploded = 0
        dark = 0
        for bitstr, cnt in counts.items():
            # bitstring order in qiskit is reverse classical reg index; invert for clarity
            bits = bitstr[::-1]
            # explosion = any mid measurement == '1'
            if '1' in bits[:zeno_steps]:
                exploded += cnt
            # define final bit 0 as 'dark' detection
            if bits[zeno_steps] == '0':
                dark += cnt
        return dark, exploded, shots, counts

    def _zeno_counts_proxy(self, shots: int, zeno_steps: int, has_ship: bool):
        # Fallback approximate model if qiskit/aer unavailable
        # Heuristic: when an object (ship) is present, dark-port click probability rises with zeno_steps
        # We'll use: p_dark = 1 - (cos(pi/(2*zeno_steps)))**(2*zeno_steps), clipped to [0, 0.999]
        # explosion probability grows with number of mid checks and small p per step
        z = max(1, zeno_steps)
        if has_ship:
            p_dark = 1.0 - (math.cos(math.pi/(2*z))**(2*z))
            p_dark = max(0.0, min(0.999, p_dark))
            # explode per step small, aggregate approx
            p_explode = min(0.5, 0.02 * z)  # very rough heuristic
        else:
            # No object: no mid-measurements; final port mostly 'bright' => final bit 1
            p_dark = 0.05  # small false dark rate
            p_explode = 0.0

        dark = 0
        exploded = 0
        for _ in range(shots):
            # decide explosion first (if has_ship)
            if has_ship and random.random() < p_explode:
                exploded += 1
            else:
                if random.random() < p_dark:
                    dark += 1
        # fabricate counts map for UI completeness
        counts = {'(proxy)': shots}
        return dark, exploded, shots, counts

    # [ADD] public probe API used by Game/UI
    def probe(self, r: int, c: int, has_ship: bool, shots: int, zeno_steps: int):
        if self.use_qiskit:
            dark, exploded, n, raw_counts = self._zeno_counts_qiskit(shots, zeno_steps, has_ship)
        else:
            dark, exploded, n, raw_counts = self._zeno_counts_proxy(shots, zeno_steps, has_ship)

        print(dark, exploded, n, raw_counts)

        # Classify
        if exploded > 0:
            label = 'explodes'
        elif dark > 0:
            label = 'found'
        else:
            label = 'no_ship'

        # measurement cost: if has_ship, we measure each step; else just final readout
        meas = shots * (zeno_steps if has_ship else 1)
        twoq = 0  # not modeling here

        p_hat = dark / n if n else 0.0
        lo, hi = self._wilson_ci(dark, n)

        outcome = Outcome(label=label, p_found=p_hat, p_explodes=(exploded / max(1, n)), counts=n)
        cost = Cost(meas=meas, twoq=twoq, shots=shots, fires=0)
        ui_stats = {
            'dark': dark,
            'exploded': exploded,
            'p_ship_ci': f'{lo:.3f}-{hi:.3f}',
            'counts': raw_counts
        }
        return outcome, cost, ui_stats


# Class Game


In [ ]:

# === Game ===
class Game:
    def __init__(self, board: Board, engine: QuantumEngine1, rules: Rules, score: Score):
        self.board = board
        self.engine = engine
        self.rules = rules
        self.score = score
        self.history = []

    def scan(self, r: int, c: int, shots: int, zeno_steps: int, side: str = "opponent"):
        # Budget check
        if self.rules.overBudget(self.score.spent):
            return Outcome("miss", 0, 0), Cost(), {'info': 'Over budget'}

        # Truth without modifying board
        truth = self.board.peek_has_ship(r, c, side=side)

        outcome, cost, stats = self.engine.probe(r, c, truth, shots, zeno_steps)
        self.score.apply(side!="opponent", outcome, cost, self.rules)

        # After scan, reveal a hint: found/miss/explodes -> mark as M unless 'found' (mark X)
        self.board.reveal(r, c, outcome.label, side=side)

        self.history.append({
            'type': 'scan', 'coord': (r, c), 'shots': shots, 'zeno_steps': zeno_steps,
            'outcome': outcome, 'cost': cost, 'stats': stats
        })
        return outcome, cost, stats

    def fire(self, r: int, c: int, side: str = "opponent"):
        if self.rules.overBudget(self.score.spent):
            return Outcome("miss", 0, 0), Cost(), {'info': 'Over budget'}

        hit = self.board.has_ship(r, c, side=side)  # modifies board
        outcome = Outcome('found' if hit else 'miss', p_found=1.0 if hit else 0.0)
        cost = Cost(meas=0, twoq=0, shots=0, fires=1)
        self.score.apply(side!="opponent", outcome, cost, self.rules)

        self.history.append({'type': 'fire', 'coord': (r, c), 'outcome': outcome, 'cost': cost})
        return outcome, cost, {}

    # === AI auto turn ===
    def ai_take_turn(self, ai: "BayesianBeliefAI"):
        # var a = bool, q = (x, y)
        # |if a == True, 跳过后面逻辑，直接fire q| - 》 原逻辑 - 》 |if scan_positive -> a = True, q 存储当前坐标|

        state = GameStateView(self, side='player')
        act = ai.suggest(state)
        f = False
        if getattr(ai,"allow_fire", False):
            outcome, cost, stats = self.fire(act.r, act.c, side='player')
        else:
            outcome, cost, stats = self.scan(act.r, act.c, act.shots, act.zeno_steps, side='player')
            f = True

        ai.update(state, Action(act.r, act.c, act.shots, act.zeno_steps), Observation(outcome, act))
        print(ai.exp_fire)
        if getattr(ai,"exp_fire", True) and f:
            outcome, cost, stats = self.fire(act.r, act.c, side='player')
            print("111")
        return {
            'board_top': self.board.opponent_board,
            'board_bottom': self.board.player_board,
            'ai_score': round(self.score.ai_points, 2),
            'user_score': None,
            'info': f"AI scanned ({act.r},{act.c}) — result: {outcome.label}",
            **stats
        }




# Class AI

In [ ]:

# === Optional: State view for AI (not wired to UI here) ===
class GameStateView:
    def __init__(self, game: Game, side: str = 'opponent'):
        self.game = game
        self.size = game.board.size
        self.board = game.board.opponent_board if side == 'opponent' else game.board.player_board

        arr = np.array(self.board, dtype=object)
        vis = np.full(arr.shape, 'unknown', dtype=object)
        vis[arr == 'M'] = 'miss'
        vis[arr == 'X'] = 'hit'
        self.visible = vis

        self.spent = game.score.spent
        self.rules = game.rules


In [ ]:
# === Bayesian AI Strategy ===
"""@dataclass
class Action:
    r: int
    c: int
    shots: int
    zeno_steps: int

@dataclass
class Observation:
    outcome: Outcome
    action: Action

class BayesianBeliefAI:
    def __init__(self, size=10, p0=0.2):
        # initial belief each cell holds a ship
        self.size = size
        self.prob = np.full((size, size), p0)

        # Likelihood table: P(result | ship), P(result | no ship)
        self.likelihood = {
            'found':     (0.95, 0.05),
            'miss':      (0.05, 0.95),
            'explodes':  (0.10, 0.01)
        }

    def suggest(self, state: GameStateView) -> Action:
        mask_unknown = (state.visible == 'unknown').astype(float)
        scores = self.prob * mask_unknown

        # If everything is known, pick highest probability anyway
        r, c = np.unravel_index(np.argmax(scores), scores.shape)

        p = self.prob[r, c]
        shots = max(50, int(500 * p))          # confident → more shots
        zeno_steps = max(5, int(30 - 25*p))    # confident → fewer steps (less explosion risk)

        return Action(r, c, shots, zeno_steps)

    def update(self, state: GameStateView, action: Action, obs: Observation):
        r, c = action.r, action.c
        prior = self.prob[r, c]

        label = obs.outcome.label
        L1, L0 = self.likelihood[label]  # P(result | ship) / P(result | no-ship)
        num = L1 * prior
        den = L1 * prior + L0 * (1 - prior)
        post = prior if den == 0 else (num / den)

        # Bayesian update
        self.prob[r, c] = post"""


"@dataclass\nclass Action:\n    r: int\n    c: int\n    shots: int\n    zeno_steps: int\n\n@dataclass\nclass Observation:\n    outcome: Outcome\n    action: Action\n\nclass BayesianBeliefAI:\n    def __init__(self, size=10, p0=0.2):\n        # initial belief each cell holds a ship\n        self.size = size\n        self.prob = np.full((size, size), p0)\n\n        # Likelihood table: P(result | ship), P(result | no ship)\n        self.likelihood = {\n            'found':     (0.95, 0.05),\n            'miss':      (0.05, 0.95),\n            'explodes':  (0.10, 0.01)\n        }\n\n    def suggest(self, state: GameStateView) -> Action:\n        mask_unknown = (state.visible == 'unknown').astype(float)\n        scores = self.prob * mask_unknown\n\n        # If everything is known, pick highest probability anyway\n        r, c = np.unravel_index(np.argmax(scores), scores.shape)\n\n        p = self.prob[r, c]\n        shots = max(50, int(500 * p))          # confident → more shots\n        z

# AI Strategy

In [ ]:
# === Bayesian AI Strategy (rewritten) ===
@dataclass
class Action:
    r: int
    c: int
    shots: int
    zeno_steps: int

@dataclass
class Observation:
    outcome: Outcome
    action: Action

class BayesianBeliefAI:
    """
    A simple Bayesian belief grid controller that mirrors the reference logic:
    - Target selection: choose among untargeted cells with the highest probability;
      break ties randomly.
    - Update rule: local, in-place updates with neighbor boost/penalty.
    """
    def __init__(self, size: int = 10, p0: float = 0.2):
        self.size = size
        self.prob = np.full((size, size), float(p0))
        self.already = set()  # type: set[tuple[int,int]]
        self.rng = np.random.default_rng()
        self.allow_fire = False
        self.exp_fire = False
    # --- helper: 4-neighborhood ---
    @staticmethod
    def _adjacent_cells(r: int, c: int, H: int, W: int):
        for rr, cc in ((r-1,c), (r+1,c), (r,c-1), (r,c+1)):
            if 0 <= rr < H and 0 <= cc < W:
                yield rr, cc

    # --- helper: choose target among max-prob untargeted cells ---
    def _choose_target(self, visible) -> tuple[int,int]:
        H, W = self.size, self.size
        max_prob = -1.0
        best = []
        for r in range(H):
            for c in range(W):
                if (r, c) in self.already:
                    continue
                if visible[r, c] != 'unknown':
                    # also skip cells already revealed by scans/fires
                    continue
                p = float(self.prob[r, c])
                if p > max_prob:
                    max_prob = p
                    best = [(r, c)]
                elif p == max_prob:
                    best.append((r, c))
        if not best:
            # fallback: pick any remaining unknown, else (0,0)
            remain = [(r,c) for r in range(H) for c in range(W)
                      if (r,c) not in self.already and visible[r,c] == 'unknown']
            if not remain:
                return (0, 0)
            idx = self.rng.integers(0, len(remain))
            return tuple(map(int, remain[idx]))
        idx = self.rng.integers(0, len(best))
        return tuple(map(int, best[idx]))

    def suggest(self, state: GameStateView) -> Action:
        r, c = self._choose_target(state.visible)
        # scale parameters gently with belief at (r,c)
        p = float(self.prob[r, c])
        shots = int(np.clip(30 + 170 * p, 10, 200))        # 10..200
        zeno_steps = int(np.clip(8 + 12 * (1 - p), 5, 20)) # 5..20
        return Action(r, c, shots, zeno_steps)

    def update(self, state: GameStateView, action: Action, obs: Observation):
        r, c = action.r, action.c
        self.already.add((r, c))

        label = obs.outcome.label  # 'found' | 'miss' | 'explodes'
        # Map to reference results
        print(label)
        if label == 'explodes':
            self.allow_fire = True
        if label == 'explodes' or label == 'found':
            self.exp_fire = True
            print('123')
        if label == 'found' or label == 'explodes':
            result = 'hit'
        elif label in 'miss':
            result = 'scan-negative'
            self.allow_fire = False
            self.exp_fire = False
        else:
            result = 'scan-negative'

        # In-place bayesian-like local update (reference rules)
        if result == 'hit':
            self.prob[r, c] = 1.0
            for rr, cc in self._adjacent_cells(r, c, self.size, self.size):
                if self.prob[rr, cc] < 1.0:
                    self.prob[rr, cc] = min(1.0, self.prob[rr, cc] + 0.3)
        elif result == 'miss':
            self.prob[r, c] = 0.0
        elif result == 'scan-positive':
            self.prob[r, c] = min(1.0, self.prob[r, c] + 0.4)
        elif result == 'scan-negative':
            self.prob[r, c] = max(0.0, self.prob[r, c] - 0.2)


# Class UI


In [ ]:

# === UI (ipywidgets) ===
from ipywidgets import Layout, GridBox, VBox, HBox, Label, Button, ToggleButtons, BoundedIntText, HTML, Output

class UserInterface:
    symbols = {
        "O": " ",
        "X": "❌",
        "M": "⭕️",
        "S": "🟨",
        "NS": "🟩",
    }

    def __init__(self, board_top, board_bottom, ships_top, title_top="AI", title_bottom="You"):
        self.board_top = board_top
        self.board_bottom = board_bottom
        self.title_top = title_top
        self.title_bottom = title_bottom
        self.ships_top = ships_top # # Ships as sets of (r, c)
                                    # self.opponent_ships: List[set] = []
        self._scan_handler = None
        self._fire_handler = None

        self._selected = None  # {'r': r, 'c': c}
        self._right_ready = False

        # Right panel outputs
        self.o_info = Output(layout=Layout(border='1px solid #ddd', min_height='50px', padding='6px'))
        self.o_stats = Output(layout=Layout(border='1px solid #ddd', min_height='80px', padding='6px'))

        # Mode toggle
        self.mode = ToggleButtons(options=['Scan', 'Fire'], value='Scan',
                                  description='Mode:', layout=Layout(width='280px'))

        # Params for Scan
        self.t_shots = BoundedIntText(value=100, min=1, max=100000, step=10, description='shots:')
        self.t_steps = BoundedIntText(value=10, min=1, max=200, step=1, description='zeno_steps:')
        self.btn_confirm = Button(description='Confirm!', button_style='success', layout=Layout(width='200px'))

        self.btn_confirm.on_click(self._on_confirm)

        # Build widgets
        self.left_top_label = HTML(value=f"<h3>{self.title_top} <span id='ai_score'></span></h3>")
        self.left_bottom_label = HTML(value=f"<h3>{self.title_bottom} <span id='user_score'></span></h3>")

        self.grid_top = self._build_grid(self.board_top, top=True)
        self.grid_bottom = self._build_grid(self.board_bottom, top=False)

        left = VBox([self.left_top_label, self.grid_top, self.left_bottom_label, self.grid_bottom],
                    layout=Layout(width='520px'))

        right_header = HTML(value="<h3>Action</h3>")
        right_mode = HBox([self.mode])
        right_params = VBox([self.t_shots, self.t_steps, self.btn_confirm],
                            layout=Layout(border='1px dashed #ccc', padding='8px', margin='4px'))
        right_panel = VBox([right_header, right_mode, right_params,
                            HTML(value="<b>Info</b>"), self.o_info,
                            HTML(value="<b>Stats</b>"), self.o_stats],
                           layout=Layout(width='520px'))

        self.widget = HBox([left, right_panel], layout=Layout(gap='16px'))

        # Add AI move button
        self.btn_ai = Button(description="AI Move", button_style='warning', layout=Layout(width='200px'))
        self.btn_ai.on_click(self._on_ai)

        right_panel.children = list(right_panel.children) + [self.btn_ai]
    def _build_grid(self, board, top: bool):
        size = len(board)
        items = []
        for r in range(size):
            row = []
            for c in range(size):
                b = Button(description=self.symbols.get(board[r][c], ' '),
                           layout=Layout(width='38px', height='38px', padding='0'),
                           tooltip=f"({r},{c})")



                b.style.button_color = '#f9f9f9'

                # print(set((r,c)), self.ships_top)

                if not top:
                    for ship in self.ships_top:
                        if (r, c) in ship:
                            b.style.button_color = None
                            b.button_style = "info"

                def _mk_onclick(rr=r, cc=c, is_top=top, btn=b):
                    def _onclick(_):
                        # Only allow clicking on top board, and only if cell is 'O'
                        if not is_top:
                            self._write_info("You can only select on the top (AI) board.")
                            return
                        if self.board_top[rr][cc] in ('M', 'X'):
                            self._write_info("Cell already resolved. Choose another.")
                            return
                        self._selected = {'r': rr, 'c': cc}
                        self._right_ready = True
                        self._write_info(f"Selected cell: ({rr}, {cc}). Choose Scan or Fire, then Confirm.")
                    return _onclick
                b.on_click(_mk_onclick())
                row.append(b)
                items.append(b)
        grid = GridBox(items, layout=Layout(grid_template_columns=' '.join(['40px']*size),
                                            grid_gap='2px'))
        return grid

    def _write_info(self, text):
        self.o_info.clear_output()
        with self.o_info:
            print(text)

    def _write_stats(self, stats: dict):
        self.o_stats.clear_output()
        with self.o_stats:
            if not stats:
                print("(no stats)")
                return
            for k in ['dark', 'exploded', 'p_ship_ci', 'counts']:
                if k in stats:
                    print(f"{k}: {stats[k]}")
            # if 'counts' in stats:
            #     print("counts:", stats['counts'])

    def _refresh_left(self):
        # Update symbols in buttons
        # top
        size = len(self.board_top)
        idx = 0
        for r in range(size):
            for c in range(size):
                btn = self.grid_top.children[idx]
                btn.description = self.symbols.get(self.board_top[r][c], ' ')
                idx += 1
        # bottom
        size2 = len(self.board_bottom)
        idx = 0
        for r in range(size2):
            for c in range(size2):
                btn = self.grid_bottom.children[idx]
                btn.description = self.symbols.get(self.board_bottom[r][c], ' ')
                idx += 1

    def set_scan_handler(self, fn):
        self._scan_handler = fn

    def set_fire_handler(self, fn):
        self._fire_handler = fn

    def _on_confirm(self, _btn):
        if not self._selected:
            self._write_info("Select a cell on the top board first.")
            return
        if self.mode.value == 'Scan':
            if not self._scan_handler:
                self._write_info("Scan handler not set.")
                return
            params = {'shots': int(self.t_shots.value), 'zeno_steps': int(self.t_steps.value)}
            resp = self._scan_handler(self._selected, params)
        else:
            if not self._fire_handler:
                self._write_info("Fire handler not set.")
                return
            resp = self._fire_handler(self._selected)

        if not isinstance(resp, dict):
            self._write_info("Handler returned non-dict response.")
            return

        # Apply updates
        self._apply_result_updates(resp)

    def _apply_result_updates(self, resp: dict):
        if 'board_top' in resp:
            self.board_top[:] = resp['board_top']
        if 'board_bottom' in resp:
            self.board_bottom[:] = resp['board_bottom']

        # refresh left boards
        self._refresh_left()

        # scores (if provided)
        ai_score = resp.get('ai_score', None)
        user_score = resp.get('user_score', None)
        ai_html = f"<h3>{self.title_top} {'— Score: ' + str(ai_score) if ai_score is not None else ''}</h3>"
        user_html = f"<h3>{self.title_bottom} {'— Score: ' + str(user_score) if user_score is not None else ''}</h3>"
        self.left_top_label.value = ai_html
        self.left_bottom_label.value = user_html

        # info / stats
        if 'info' in resp:
            self._write_info(resp['info'])
        stats = {k: resp[k] for k in ['dark', 'exploded', 'p_ship_ci', 'counts'] if k in resp}
        self._write_stats(stats)

        print(f"Stats: AI: {Score.ai_usage} User: {Score.user_usage}")


    def set_ai_handler(self, fn):
        self._ai_handler = fn

    def _on_ai(self, _):
        if not hasattr(self, "_ai_handler") or self._ai_handler is None:
            self._write_info("AI handler not set.")
            return
        resp = self._ai_handler()
        if isinstance(resp, dict):
            self._apply_result_updates(resp)



# Run

In [ ]:

# === Driver: Build world, bind handlers, display UI ===
from IPython.display import display

# Build a game
board = Board(size=10)
board.place_ship(side="opponent", seed=random.randint(1, 10000), count=6, length=4)
board.place_ship(side="player", seed=random.randint(1, 10000), count=4, length=4)  # optional; shows on your side

engine = QuantumEngine1(real_backend=False)  # set True to use qiskit-aer if installed
rules = Rules()
score = Score()
game = Game(board=board, engine=engine, rules=rules, score=score)

ui = UserInterface(board_top=board.opponent_board, board_bottom=board.player_board,
                   ships_top=board.player_ships, title_top="AI", title_bottom="You")

def on_scan(selected, params):
    r, c = selected['r'], selected['c']
    shots = int(params['shots'])
    steps = int(params['zeno_steps'])
    outcome, cost, stats = game.scan(r, c, shots, steps, side='opponent')
    print(f"Stats: AI: {Score.ai_usage} User: {Score.user_usage}")
    return {
        'board_top': game.board.opponent_board,
        'board_bottom': game.board.player_board,
        'ai_score': round(game.score.ai_points, 2),
        'user_score': round(game.score.user_points, 2),
        **stats
    }

def on_fire(selected):
    r, c = selected['r'], selected['c']
    outcome, cost, stats = game.fire(r, c, side='opponent')
    print(f"Stats: AI: {Score.ai_usage} User: {Score.user_usage}")
    return {
        'board_top': game.board.opponent_board,
        'board_bottom': game.board.player_board,
        'ai_score': round(game.score.ai_points, 2),
        'user_score': round(game.score.user_points, 2),
        'info': ('HIT!' if outcome.label == 'found' else 'MISS')
    }

ui.set_scan_handler(on_scan)
ui.set_fire_handler(on_fire)

display(ui.widget)
print("Ready. Select a cell on the top board, choose Scan or Fire, then click Confirm!")

# === Init AI ===
ai = BayesianBeliefAI(size=10)

# === Bind AI button to game logic ===
def on_ai_move():
    return game.ai_take_turn(ai)

ui.set_ai_handler(on_ai_move)

print("AI ready. You can scan/fire, or click 'AI Move' to let AI act.")


Ready. Select a cell on the top board, choose Scan or Fire, then click Confirm!
AI ready. You can scan/fire, or click 'AI Move' to let AI act.
0 0 100 {'10000000000': 100}
Stats: AI: {'shots': 320, 'measuments': 320, 'positives': 0.0, 'explosions': 0.0, 'fire times': 0, 'hits': 0, 'misses': 0} User: {'shots': 400, 'measuments': 1300, 'positives': 100.0, 'explosions': 18.0, 'fire times': 3, 'hits': 2.0, 'misses': 0.0}
Stats: AI: {'shots': 320, 'measuments': 320, 'positives': 0.0, 'explosions': 0.0, 'fire times': 0, 'hits': 0, 'misses': 0} User: {'shots': 400, 'measuments': 1300, 'positives': 100.0, 'explosions': 18.0, 'fire times': 3, 'hits': 2.0, 'misses': 0.0}
0 0 100 {'10000000000': 100}
Stats: AI: {'shots': 320, 'measuments': 320, 'positives': 0.0, 'explosions': 0.0, 'fire times': 0, 'hits': 0, 'misses': 0} User: {'shots': 500, 'measuments': 1400, 'positives': 100.0, 'explosions': 18.0, 'fire times': 3, 'hits': 2.0, 'misses': 0.0}
Stats: AI: {'shots': 320, 'measuments': 320, 'positi